<a href="https://colab.research.google.com/github/jzazueta/data-science-business-econ/blob/main/notebooks/ch10_timeseries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 10 — Time Series: From Cross-Section to Forecast

**Data Science for Business and Economics: A Python Approach**---Jorge Zazueta Gutiérrez

This notebook mirrors every code example in Chapter 10. Tomás Ibarra, a demand planner at
Lácteos del Bajío in Querétaro, has to commit a production quantity for drinkable yogurt ten
days before the week it ships.

Run the cells in order — later cells depend on names defined earlier.

### Setup — getting the data and the packages

These two cells are notebook scaffolding and do not appear in the book. The first finds the two
extracts in the repository, or downloads them if you are running in Colab. The second installs
`prophet`, which Section 10.5 needs and which Colab does not ship by default.

In [1]:
import os
import urllib.request

RAW = ("https://raw.githubusercontent.com/jzazueta/data-science-business-econ/"
       "main/data/")

for name in ("lacteos_weekly_demand.csv", "consumer_spending_index.csv"):
    if os.path.exists(name):
        continue
    for candidate in ("data/" + name, "../data/" + name):
        if os.path.exists(candidate):
            os.link(candidate, name)
            break
    else:
        urllib.request.urlretrieve(RAW + name, name)

print(os.listdir("."))

['lacteos_weekly_demand.csv', 'consumer_spending_index.csv']


In [2]:
try:
    import prophet
except ImportError:
    %pip install -q prophet
    import prophet

print(prophet.__version__)

1.4.0


## 10.1 From Rows to Sequences

Tomás Ibarra plans weekly production of drinkable yogurt at Lácteos del Bajío. Every Thursday he commits a quantity for the week after next. This section asks what changes when the rows of a dataset are ordered in time.

### Code Example 10.1 — Loading the weekly shipments extract

In [3]:
import numpy as np
import pandas as pd

# Read the extract; parse_dates turns the text column into
# real dates that pandas can sort, group and compare
demand = pd.read_csv("lacteos_weekly_demand.csv",
                     parse_dates=["week_start"])
print(demand.shape)                 # (rows, columns)
print(demand["product_line"].value_counts())

# First and last week in the file
weeks = demand["week_start"]
print(weeks.min().date(), "to", weeks.max().date())

(1040, 6)
product_line
drinkable_yogurt    260
fresh_cream         260
panela_cheese       260
flavored_milk       260
Name: count, dtype: int64
2021-01-04 to 2025-12-22


### Code Example 10.2 — Isolating the yogurt series and averaging it by year

In [4]:
# Keep the yogurt rows and make the week the index, so the
# result is a series of units labeled by date
yogurt = (demand[demand["product_line"] == "drinkable_yogurt"]
          .set_index("week_start")["units"])
print(yogurt.head())

# Average weekly units within each calendar year
print(yogurt.groupby(yogurt.index.year).mean().round(0))

week_start
2021-01-04    19533
2021-01-11    20185
2021-01-18    25344
2021-01-25    19013
2021-02-01    19467
Name: units, dtype: int64
week_start
2021    19088.0
2022    21166.0
2023    22198.0
2024    25106.0
2025    25943.0
Name: units, dtype: float64


### Code Example 10.3 — Bringing the weekly series to the index's monthly frequency

In [5]:
index = pd.read_csv("consumer_spending_index.csv",
                    parse_dates=["month"])
index.columns = ["month", "spending"]   # shorter name

# resample("MS") groups the weeks by calendar month, each
# group labeled by the month's first day; size() counts the
# weeks that start in each month
print(yogurt.resample("MS").size().value_counts())

# Average weekly units per month, joined to the index
monthly = (yogurt.resample("MS").mean()
           .rename("units").rename_axis("month").to_frame()
           .join(index.set_index("month")))
print(monthly.shape)
print(monthly.head(3).round(2))

units
4    40
5    20
Name: count, dtype: int64
(60, 2)
               units  spending
month                         
2021-01-01  21018.75    100.00
2021-02-01  21535.00    101.79
2021-03-01  19884.80    102.36


### Code Example 10.4 — The same regression in levels and in monthly changes

In [6]:
import statsmodels.api as sm

log_m = np.log(monthly)
# diff() subtracts the previous month from each month: in
# logs, the result is each month's growth rate
diffs = log_m.diff().dropna()

# The same regression, first in levels, then in changes
for name, data in [("levels", log_m), ("changes", diffs)]:
    fit = sm.OLS(data["units"],
                 sm.add_constant(data["spending"])).fit()
    print("%-8s slope %5.2f   t %5.2f   R2 %.4f"
          % (name, fit.params["spending"],
             fit.tvalues["spending"], fit.rsquared))

levels   slope  1.07   t  7.00   R2 0.4578
changes  slope  0.20   t  0.14   R2 0.0003


## 10.2 Decomposing a Series

Splitting the series into a trend, a seasonal pattern and what is left over, on the log scale, with Fourier terms and indicator variables for the events a smooth curve cannot place.

### Code Example 10.5 — Checking whether variation grows with the level

In [7]:
# Mean and standard deviation of weekly units, by year
by_year = (yogurt.groupby(yogurt.index.year)
           .agg(["mean", "std"]))
# Variation relative to the level of the series
by_year["std / mean"] = by_year["std"] / by_year["mean"]
print(by_year.round({"mean": 0, "std": 0, "std / mean": 3}))

               mean     std  std / mean
week_start                             
2021        19088.0  2755.0       0.144
2022        21166.0  3210.0       0.152
2023        22198.0  3756.0       0.169
2024        25106.0  3758.0       0.150
2025        25943.0  3582.0       0.138


### Code Example 10.6 — A centered 52-week moving average as the trend

In [8]:
log_y = np.log(yogurt)

# Average of the 52 weeks around each week; center=True
# places the week in the middle of its window
trend_ma = log_y.rolling(52, center=True).mean()

print("weeks without a trend value:", trend_ma.isna().sum())
print("last week with one:",
      trend_ma.last_valid_index().date())

# Log growth between the first and last trend values,
# converted to a yearly percentage
tr = trend_ma.dropna()
per_year = (tr.iloc[-1] - tr.iloc[0]) * 52 / (len(tr) - 1)
print("trend growth per year: %.4f" % (np.exp(per_year) - 1))

weeks without a trend value: 51
last week with one: 2025-06-30
trend growth per year: 0.0805


### Code Example 10.7 — Averaging the detrended series by week of year

In [9]:
# Remove the trend; what is left is seasonal + remainder
detrended = log_y - trend_ma
week = log_y.index.isocalendar().week   # week of year, 1-52

# Count and average of the detrended values for each week
by_week = detrended.groupby(week).agg(["count", "mean"])
print(by_week["count"].value_counts())
print("range of weekly means:     %.3f"
      % (by_week["mean"].max() - by_week["mean"].min()))
# Typical size of the jump from one week's mean to the next
print("typical week-to-week jump: %.3f"
      % by_week["mean"].diff().std())

count
4    51
5     1
Name: count, dtype: int64
range of weekly means:     0.430
typical week-to-week jump: 0.108


### Code Example 10.8 — A seasonal component from two harmonics

In [10]:
import statsmodels.api as sm

def fourier_terms(index, K):
    # Position of each week in the year, as an angle that
    # goes once around the circle (0 to 2*pi) per year
    angle = 2 * np.pi * index.dayofyear / 365.25
    terms = {}
    for k in range(1, K + 1):
        # Harmonic k completes k cycles per year
        terms[f"sin{k}"] = np.sin(k * angle)
        terms[f"cos{k}"] = np.cos(k * angle)
    return pd.DataFrame(terms, index=index)

waves = fourier_terms(log_y.index, K=2)
known = detrended.notna()     # weeks with a trend value

# Regress the detrended series on the four wave columns
wave_fit = sm.OLS(detrended[known],
                  sm.add_constant(waves[known])).fit()
seasonal = wave_fit.predict(sm.add_constant(waves))

print("typical week-to-week jump: %.3f"
      % seasonal.diff().std())
print("R-squared: %.4f" % wave_fit.rsquared)

typical week-to-week jump: 0.010
R-squared: 0.2570


### Code Example 10.9 — The week of the year in which Semana Santa fell

In [11]:
# The three indicator columns for the yogurt line
flags = (demand[demand["product_line"] == "drinkable_yogurt"]
         .set_index("week_start")[["promo", "semana_santa",
                                   "pascua"]])

# Week of the year in which each Semana Santa fell
santa = flags.index[flags["semana_santa"] == 1]
print(pd.Series(santa.isocalendar().week.to_numpy(),
                index=santa.year, name="week of year"))

week_start
2021    13
2022    15
2023    14
2024    13
2025    16
Name: week of year, dtype: uint32


### Code Example 10.10 — Fourier terms and event indicators in one regression

In [12]:
# Predictors: constant, four wave columns, three indicators
X = sm.add_constant(waves.join(flags))
cal_fit = sm.OLS(detrended[known], X[known]).fit()

print(cal_fit.params[["const", "promo", "semana_santa",
                      "pascua"]].round(4))
print("R-squared: %.4f" % cal_fit.rsquared)

const          -0.0403
promo           0.2852
semana_santa   -0.2023
pascua         -0.0859
dtype: float64
R-squared: 0.7994


### Code Example 10.11 — The same regression with four harmonics

In [13]:
# Same regression with four harmonics (eight wave columns)
waves4 = fourier_terms(log_y.index, K=4)
X4 = sm.add_constant(waves4.join(flags))
cal_fit4 = sm.OLS(detrended[known], X4[known]).fit()

print(cal_fit4.params[["promo", "semana_santa",
                       "pascua"]].round(4))
print("R-squared: %.4f" % cal_fit4.rsquared)

promo           0.2844
semana_santa   -0.1929
pascua         -0.0752
dtype: float64
R-squared: 0.8330


### Code Example 10.12 — Separating the remainder

In [14]:
p = cal_fit4.params
# Fitted seasonal curve: wave columns times their weights
seasonal = waves4.dot(p[waves4.columns])
# Fitted event effects: indicators times their weights
events = flags.dot(p[flags.columns])
remainder = detrended - p["const"] - seasonal - events

print("remainder std. dev.: %.4f" % remainder.std())

remainder std. dev.: 0.0624


### Code Example 10.13 — Pairing each week with the previous one, and the autocorrelations

In [15]:
from statsmodels.tsa.stattools import acf

# shift(1) moves every value one week later, so each week
# is lined up with the previous week's remainder
pairs = pd.DataFrame({"this week": remainder,
                      "last week": remainder.shift(1)})
print(pairs.dropna().head(3).round(4))

# Autocorrelations at lags 1 to 8
r = remainder.dropna()
print(pd.Series(acf(r, nlags=8)[1:], index=range(1, 9),
                name="acf").round(3))
print("band: +/- %.3f" % (2 / np.sqrt(len(r))))

            this week  last week
week_start                      
2021-07-12     0.0003     0.0252
2021-07-19    -0.1000     0.0003
2021-07-26     0.0136    -0.1000
1    0.565
2    0.350
3    0.298
4    0.183
5    0.166
6    0.200
7    0.167
8    0.116
Name: acf, dtype: float64
band: +/- 0.138


## 10.3 Benchmarks and Honest Evaluation

Forecasts are only comparable under rules: a forecast origin, a horizon, an error measure and benchmarks to beat. Evaluation here rolls forward one week at a time.

### Code Example 10.14 — The evaluation period and two error measures

In [16]:
HORIZON = 2       # weeks between origin and target
test_weeks = yogurt.index[yogurt.index.year == 2025]

def mae(actual, forecast):
    # Average size of the errors, in units
    return np.mean(np.abs(actual - forecast))

def mape(actual, forecast):
    # Average size of the errors, as a percent of actual
    return 100 * np.mean(np.abs(actual - forecast) / actual)

actual = yogurt[test_weeks]
print(len(test_weeks), "weeks from", test_weeks[0].date(),
      "to", test_weeks[-1].date())

51 weeks from 2025-01-06 to 2025-12-22


### Code Example 10.15 — The naive and seasonal naive benchmarks

In [17]:
# shift(k) moves the series k weeks later: each week then
# holds the value observed k weeks earlier
benchmarks = {
    "naive": yogurt.shift(HORIZON)[test_weeks],
    "seasonal naive": yogurt.shift(52)[test_weeks],
}
for name, fc in benchmarks.items():
    print("%-15s MAE %6.0f   MAPE %5.2f"
          % (name, mae(actual, fc), mape(actual, fc)))

naive           MAE   3299   MAPE 11.94
seasonal naive  MAE   3169   MAPE 11.67


### Code Example 10.16 — Rolling-origin forecasts from the calendar regression

In [18]:
def calendar_design(K):
    # Waves and indicators, plus time in years for the trend
    X = fourier_terms(log_y.index, K).join(flags)
    X.insert(0, "years", np.arange(len(X)) / 52)
    return sm.add_constant(X)

def rolling_forecast(X):
    forecasts = []
    for week in test_weeks:
        target = log_y.index.get_loc(week)   # row of target
        origin = target - HORIZON            # last row known
        # Fit on rows up to and including the origin only
        fit = sm.OLS(log_y.iloc[:origin + 1],
                     X.iloc[:origin + 1]).fit()
        pred = fit.predict(X.iloc[[target]])
        forecasts.append(pred.iloc[0])
    # Back from logs to units
    return pd.Series(np.exp(forecasts), index=test_weeks)

for K in (2, 4):
    fc = rolling_forecast(calendar_design(K))
    benchmarks[f"calendar, K={K}"] = fc
    print("calendar, K=%d    MAE %6.0f   MAPE %5.2f"
          % (K, mae(actual, fc), mape(actual, fc)))

calendar, K=2    MAE   1490   MAPE  5.58


calendar, K=4    MAE   1290   MAPE  4.88

### Code Example 10.17 — How the calendar regression errs

In [19]:
# Error in logs: negative when the forecast was too high
errors = np.log(actual) - np.log(benchmarks["calendar, K=4"])
q1 = errors.index.quarter == 1       # January to March

print("weeks over-forecast in Q1:", (errors[q1] < 0).sum(),
      "of", q1.sum())
print("mean log error, Q1:    %.4f" % errors[q1].mean())
print("mean log error, Q2-Q4: %.4f" % errors[~q1].mean())
print("lag-1 autocorrelation of errors: %.4f"
      % errors.autocorr(1))

weeks over-forecast in Q1: 13 of 13
mean log error, Q1:    -0.0932
mean log error, Q2-Q4: 0.0036
lag-1 autocorrelation of errors: 0.5702


## 10.4 Autocorrelation and ARIMA Models

What the calendar model leaves over is not noise: a miss in one week predicts a miss in the next. An autoregressive term puts that persistence to work.

### Code Example 10.18 — Autocorrelation and partial autocorrelation of the residuals

In [20]:
from statsmodels.tsa.stattools import pacf

# Calendar regression fitted on the weeks before 2025 only
X4 = calendar_design(4)
before = log_y.index < test_weeks[0]
cal_resid = sm.OLS(log_y[before], X4[before]).fit().resid

# Autocorrelation and partial autocorrelation, lags 1 to 6
print(pd.DataFrame({"acf": acf(cal_resid, nlags=6)[1:],
                    "pacf": pacf(cal_resid, nlags=6)[1:]},
                   index=range(1, 7)).round(3))

     acf   pacf
1  0.624  0.627
2  0.408  0.030
3  0.358  0.156
4  0.288  0.006
5  0.301  0.149
6  0.345  0.134


### Code Example 10.19 — A calendar regression with AR(1) errors

In [21]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Declare the weekly frequency: one row per Monday
y_w = log_y.asfreq("W-MON")
X_w = X4.asfreq("W-MON")

# Calendar regression whose errors follow an AR(1)
model = SARIMAX(y_w[before], exog=X_w[before],
                order=(1, 0, 0))
ar1 = model.fit(disp=False, method="powell", maxiter=5000)
print(ar1.params[["years", "promo", "semana_santa",
                  "ar.L1", "sigma2"]].round(4))

years           0.0873
promo           0.3075
semana_santa   -0.2083
ar.L1           0.6475
sigma2          0.0023
dtype: float64


### Code Example 10.20 — Rolling-origin forecasts with AR corrections

In [22]:
def rolling_arima(order):
    forecasts = []
    for week in test_weeks:
        target = y_w.index.get_loc(week)
        origin = target - HORIZON
        # Refit on the weeks up to the origin
        model = SARIMAX(y_w.iloc[:origin + 1], order=order,
                        exog=X_w.iloc[:origin + 1])
        fit = model.fit(disp=False, method="powell",
                        maxiter=5000)
        # Calendar values for the next two weeks are known
        future = X_w.iloc[origin + 1:target + 1]
        path = fit.forecast(HORIZON, exog=future)
        forecasts.append(path.iloc[-1])   # two weeks ahead
    return pd.Series(np.exp(forecasts), index=test_weeks)

for order in [(1, 0, 0), (2, 0, 0)]:
    fc = rolling_arima(order)
    benchmarks[f"calendar + AR({order[0]})"] = fc
    print("calendar + AR(%d)  MAE %6.0f   MAPE %5.2f"
          % (order[0], mae(actual, fc), mape(actual, fc)))

calendar + AR(1)  MAE   1155   MAPE  4.34


calendar + AR(2)  MAE   1162   MAPE  4.35


### Code Example 10.21 — How the corrected forecasts err

In [23]:
errors_ar = (np.log(actual)
             - np.log(benchmarks["calendar + AR(1)"]))

print("mean log error, Q1:    %.4f" % errors_ar[q1].mean())
print("mean log error, Q2-Q4: %.4f" % errors_ar[~q1].mean())
print("lag-1 autocorrelation of errors: %.4f"
      % errors_ar.autocorr(1))

mean log error, Q1:    -0.0671
mean log error, Q2-Q4: 0.0072
lag-1 autocorrelation of errors: 0.3518


## 10.5 Exponential Smoothing and Prophet

Two methods that learn the pattern from the series itself, and what happens when they are given the same calendar the regression had.

### Code Example 10.22 — Exponential smoothing, with and without the calendar

In [24]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

X_cal = X4.drop(columns="years")     # calendar, no trend

def rolling_smoothing(adjust):
    forecasts, alphas = [], []
    for week in test_weeks:
        target = log_y.index.get_loc(week)
        origin = target - HORIZON
        calendar = pd.Series(0.0, index=log_y.index)
        if adjust:
            # Calendar effects estimated on data to the origin
            fit = sm.OLS(log_y.iloc[:origin + 1],
                         X_cal.iloc[:origin + 1]).fit()
            effects = fit.params.drop("const")
            calendar = X_cal[effects.index].dot(effects)
        # Smooth what the calendar does not explain
        adjusted = (log_y - calendar).iloc[:origin + 1]
        ses = SimpleExpSmoothing(
            adjusted.to_numpy(),
            initialization_method="estimated").fit()
        alphas.append(ses.params["smoothing_level"])
        # Forecast = smoothed level + calendar for the target
        level = ses.forecast(HORIZON)[-1]
        forecasts.append(level + calendar.iloc[target])
    fc = pd.Series(np.exp(forecasts), index=test_weeks)
    return fc, np.mean(alphas)

for adjust in (False, True):
    fc, alpha = rolling_smoothing(adjust)
    name = "smoothing" + (" + calendar" if adjust else "")
    benchmarks[name] = fc
    print("%-21s MAE %5.0f  MAPE %5.2f  alpha %.2f"
          % (name, mae(actual, fc), mape(actual, fc), alpha))

smoothing             MAE  2661  MAPE  9.75  alpha 0.23


smoothing + calendar  MAE  1205  MAPE  4.59  alpha 0.51


### Code Example 10.23 — Rolling-origin forecasts from Prophet

In [25]:
import logging
from prophet import Prophet

# Silence the progress messages Prophet prints on each fit
logging.getLogger("cmdstanpy").disabled = True

# Prophet expects a date column "ds" and a value column "y"
frame = pd.DataFrame({"ds": log_y.index,
                      "y": log_y.to_numpy()})
frame = frame.join(flags.reset_index(drop=True))

def rolling_prophet(flexibility):
    forecasts = []
    for week in test_weeks:
        target = log_y.index.get_loc(week)
        origin = target - HORIZON
        m = Prophet(yearly_seasonality=4,      # harmonics
                    weekly_seasonality=False,
                    daily_seasonality=False,
                    changepoint_prior_scale=flexibility)
        for col in flags.columns:
            m.add_regressor(col)               # indicators
        m.fit(frame.iloc[:origin + 1])
        pred = m.predict(frame.iloc[[target]])
        forecasts.append(pred["yhat"].iloc[0])
    return pd.Series(np.exp(forecasts), index=test_weeks)

for flexibility in (0.05, 0.001):
    fc = rolling_prophet(flexibility)
    benchmarks[f"Prophet, {flexibility}"] = fc
    print("Prophet, %-6s  MAE %5.0f  MAPE %5.2f"
          % (flexibility, mae(actual, fc), mape(actual, fc)))

Prophet, 0.05    MAE  1649  MAPE  6.16


Prophet, 0.001   MAE  1288  MAPE  4.87


### Code Example 10.24 — All forecasts on the same evaluation weeks

In [26]:
# One row per forecasting method, best first
table = pd.DataFrame(
    {name: {"MAE": mae(actual, fc), "MAPE": mape(actual, fc)}
     for name, fc in benchmarks.items()}).T
print(table.sort_values("MAPE").round({"MAE": 0, "MAPE": 2}))

                         MAE   MAPE
calendar + AR(1)      1155.0   4.34
calendar + AR(2)      1162.0   4.35
smoothing + calendar  1205.0   4.59
Prophet, 0.001        1288.0   4.87
calendar, K=4         1290.0   4.88
calendar, K=2         1490.0   5.58
Prophet, 0.05         1649.0   6.16
smoothing             2661.0   9.75
seasonal naive        3169.0  11.67
naive                 3299.0  11.94


## 10.6 From Forecast to Decision

A forecast distribution, a cost of being short, a cost of being long, and the order quantity that follows from them.

### Code Example 10.25 — The forecast distribution, not just its center

In [27]:
from scipy.stats import norm

def rolling_distribution(order=(1, 0, 0)):
    means, sds = [], []
    for week in test_weeks:
        target = y_w.index.get_loc(week)
        origin = target - HORIZON
        model = SARIMAX(y_w.iloc[:origin + 1], order=order,
                        exog=X_w.iloc[:origin + 1])
        fit = model.fit(disp=False, method="powell",
                        maxiter=5000)
        # get_forecast keeps the uncertainty, not just the
        # central path that forecast() returns
        step = fit.get_forecast(
            HORIZON, exog=X_w.iloc[origin + 1:target + 1])
        means.append(step.predicted_mean.iloc[-1])
        sds.append(step.se_mean.iloc[-1])
    return pd.DataFrame({"mu": means, "sigma": sds},
                        index=test_weeks)

pred = rolling_distribution()
print(pred.head(3).round(4))
print("average sigma: %.4f" % pred["sigma"].mean())

                 mu   sigma
week_start                 
2025-01-06  10.5728  0.0569
2025-01-13  10.2445  0.0567
2025-01-20  10.1528  0.0583
average sigma: 0.0579


### Code Example 10.26 — Do the intervals cover what they claim?

In [28]:
def quantile_order(p):
    # Quantile p of the lognormal forecast distribution
    return np.exp(pred["mu"] + norm.ppf(p) * pred["sigma"])

for level in (0.50, 0.80, 0.95):
    lo = quantile_order(0.5 - level / 2)
    hi = quantile_order(0.5 + level / 2)
    inside = ((actual >= lo) & (actual <= hi)).mean()
    print("%.0f%% interval: covers %.0f%% of weeks,"
          " average width %5.0f units"
          % (100 * level, 100 * inside, (hi - lo).mean()))

50% interval: covers 55% of weeks, average width  2056 units
80% interval: covers 86% of weeks, average width  3908 units
95% interval: covers 92% of weeks, average width  5985 units


### Code Example 10.27 — The cost of a unit short and a unit unsold

In [29]:
PRICE = 22      # pesos per unit shipped to the retailer
COST = 13       # pesos to produce one unit

short_cost = PRICE - COST    # margin lost on a unit short
excess_cost = COST           # a unit unsold is written off
ratio = short_cost / (short_cost + excess_cost)
print("critical ratio: %.3f" % ratio)

def total_cost(order):
    # Shortfalls cost margin; excess costs production
    short = np.maximum(actual - order, 0) * short_cost
    excess = np.maximum(order - actual, 0) * excess_cost
    return (short + excess).sum()

print("order at the median:  %8.0f pesos"
      % total_cost(quantile_order(0.5)))
print("order at the ratio:   %8.0f pesos"
      % total_cost(quantile_order(ratio)))
print("average units ordered: %.0f at the median,"
      " %.0f at the ratio"
      % (quantile_order(0.5).mean(),
         quantile_order(ratio).mean()))

critical ratio: 0.409
order at the median:    686509 pesos
order at the ratio:     668852 pesos
average units ordered: 26324 at the median, 25976 at the ratio


### Code Example 10.28 — What every quantile would have cost

In [30]:
# Cost of ordering at each quantile from 5% to 95%
grid = np.arange(0.05, 0.96, 0.025)
costs = pd.Series([total_cost(quantile_order(p))
                   for p in grid], index=grid.round(3))

print("cheapest quantile here: %.3f" % costs.idxmin())
print("its cost: %.0f pesos" % costs.min())
# What last year's shipments would have cost as an order
naive_order = yogurt.shift(52)[test_weeks]
print("seasonal naive order: %.0f pesos"
      % total_cost(naive_order))

cheapest quantile here: 0.425
its cost: 668888 pesos
seasonal naive order: 1690408 pesos


## Where this leaves Tomás

Repeating last year's week misses by 11.67 percent. The calendar regression of Section 10.3
misses by 4.88, the AR(1) correction of Section 10.4 by 4.34, and neither exponential smoothing
nor Prophet beats that once all three are given the same information. Ordering at the 0.409
quantile of the forecast distribution rather than at last year's shipments cuts the cost of a
year of production decisions by 60 percent.

The exercises at the end of the chapter continue from here: 12-month differences, an STL
decomposition, a differenced ARIMA, the other three product lines, and a different set of costs.